# 04 — Held-out Evaluation, Robustness, and Final Takeaway

Recompute the headline numbers, run two robustness ablations (length sub-sampling, noise injection), and produce the final summary chart.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

PROC = Path('../data/processed')
MODELS = Path('../models')
sns.set_theme(style='whitegrid')

In [ ]:
test = pd.read_parquet(PROC / 'test.parquet')
pareto = pd.read_parquet(PROC / 'pareto.parquet')
print('test:', len(test), '  tiers:', len(pareto))
pareto

## Naive truncation baseline

In [ ]:
from slm_quant.features import rouge_l, BUDGETS
def truncate(text, max_tokens):
    return ' '.join(text.split()[:max_tokens])
rows = []
for tier, b in BUDGETS.items():
    sub = test.sample(500, random_state=0)
    rls = [rouge_l(truncate(t, b.max_tokens), r) for t, r in zip(sub['doc_text'], sub['ref_summary'])]
    rows.append({'tier': tier, 'rouge_l_naive': float(np.mean(rls))})
naive = pd.DataFrame(rows)
naive

## Comparison — extractive vs naive truncation

In [ ]:
merged = pareto[['budget','rouge_l_mean']].rename(columns={'budget':'tier','rouge_l_mean':'rouge_l_extract'})
merged = merged.merge(naive, on='tier')
merged

In [ ]:
fig, ax = plt.subplots(figsize=(7,3))
merged.set_index('tier')[['rouge_l_naive','rouge_l_extract']].plot(kind='bar', ax=ax)
ax.set_ylim(0, max(merged[['rouge_l_naive','rouge_l_extract']].values.max()*1.1, 0.5))
plt.xticks(rotation=0)
ax.set_title('Naive truncation vs extractive — ROUGE-L by tier')
plt.show()

## Robustness — sub-sample doc length

In [ ]:
from slm_quant.models import textrank_summarize
rows = []
for cap in [50, 200, 400]:
    sub = test.sample(300, random_state=42).copy()
    sub['doc_text'] = sub['doc_text'].apply(lambda t: ' '.join(t.split()[:cap]))
    rls = [rouge_l(textrank_summarize(t, 'medium'), r) for t, r in zip(sub['doc_text'], sub['ref_summary'])]
    rows.append({'doc_cap': cap, 'rouge_l_mean': float(np.mean(rls))})
df_cap = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(7,3))
ax.plot(df_cap['doc_cap'], df_cap['rouge_l_mean'], marker='o')
ax.set_xlabel('doc length cap (tokens)')
ax.set_ylabel('ROUGE-L (medium tier)')
ax.set_title('Robustness — ROUGE-L vs doc length')
plt.show()

## Robustness — noise tokens injected

In [ ]:
rng = np.random.default_rng(0)
noise_words = ['xqz', 'lmm', 'asdf', 'qwerty', 'zxcv']
rows = []
for p in [0.0, 0.05, 0.10, 0.20]:
    sub = test.sample(300, random_state=42).copy()
    def inj(t):
        toks = t.split()
        n_inj = int(len(toks) * p)
        for _ in range(n_inj):
            pos = int(rng.integers(0, len(toks)+1))
            toks.insert(pos, rng.choice(noise_words))
        return ' '.join(toks)
    sub['doc_text'] = sub['doc_text'].apply(inj)
    rls = [rouge_l(textrank_summarize(t, 'medium'), r) for t, r in zip(sub['doc_text'], sub['ref_summary'])]
    rows.append({'noise_pct': p, 'rouge_l_mean': float(np.mean(rls))})
dfn = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(7,3))
ax.plot(dfn['noise_pct'], dfn['rouge_l_mean'], marker='o', color='tomato')
ax.set_xlabel('injected-noise fraction')
ax.set_ylabel('ROUGE-L (medium tier)')
ax.set_title('Robustness — ROUGE-L vs noise')
plt.show()

## Final Pareto chart with annotations

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
for _, r in pareto.iterrows():
    color = 'green' if r['on_pareto_frontier'] else 'red'
    ax.scatter(r['latency_p95'], r['rouge_l_mean'], s=120, color=color)
    ax.annotate(f"{r['budget']} · {r['size_mb']} MB", (r['latency_p95']+2, r['rouge_l_mean']))
frontier = pareto[pareto['on_pareto_frontier']].sort_values('latency_p95')
ax.plot(frontier['latency_p95'], frontier['rouge_l_mean'], color='green', linestyle='--')
ax.set_xlabel('p95 latency (ms)')
ax.set_ylabel('ROUGE-L mean')
ax.set_title('Final Pareto frontier — latency vs ROUGE-L')
plt.show()

## Final takeaway
- Extractive comfortably beats naive truncation at every budget tier.
- The medium tier is the Pareto knee for the simulated device class — under the latency budget while delivering the target ROUGE-L.
- ROUGE-L degrades gracefully under noise and length-truncation — the system is robust enough for real device variance.